In [2]:
import pandas as pd

df = pd.read_csv("data/processed/pincode_baseline_stats.csv")
df.head(20)


,pincode,median_monthly_total,q1_monthly_total,q3_monthly_total,median_bio,median_demo,median_enroll,iqr_monthly_total,bio_demo_ratio,enroll_demo_ratio
0,400601,1821.0,1049.00,2308.00,1315.0,521.0,77.0,1259.0,2.523992,0.147793
1,400602,494.0,319.50,693.50,361.0,28.0,5.0,374.0,12.892857,0.178571
2,400603,665.0,395.00,987.50,543.0,13.0,2.0,592.5,41.769231,0.153846
3,400604,2524.0,1851.50,3609.50,2050.0,151.0,27.0,1758.0,13.576159,0.178808
4,400605,3119.0,2014.50,3940.50,2341.0,683.0,31.0,1926.0,3.427526,0.045388
5,400606,2142.0,1042.50,3209.00,1314.0,375.0,18.0,2166.5,3.504000,0.048000
6,400607,1925.0,980.00,2357.50,918.0,523.0,70.0,1377.5,1.755258,0.133843
7,400608,408.0,145.50,462.50,164.0,21.0,10.0,317.0,7.809524,0.476190
8,400610,508.0,417.50,1000.00,485.0,47.0,6.0,582.5,10.319149,0.127660
9,400612,9440.0,8584.50,12542.50,5901.0,4189.0,534.0,3958.0,1.408689,0.127477


In [3]:
import pandas as pd

df = pd.read_csv("data/processed/monthly_pin_level.csv")

month_counts = (
    df.groupby('pincode')['month']
      .nunique()
      .reset_index(name='num_months')
      .sort_values('num_months')
)

month_counts.head(20)


,pincode,num_months
57,401609,6
56,401608,7
21,401103,9
30,401206,9
53,401605,9
58,401610,9
22,401104,9
49,401601,9
46,401503,9
41,401403,9


In [4]:
zero_ratio = (
    df.assign(is_zero = df['total_monthly_count'] == 0)
      .groupby('pincode')['is_zero']
      .mean()
      .reset_index(name='zero_month_ratio')
      .sort_values('zero_month_ratio', ascending=False)
)

zero_ratio.head(20)


,pincode,zero_month_ratio
0,400601,0.0
1,400602,0.0
2,400603,0.0
3,400604,0.0
4,400605,0.0
5,400606,0.0
6,400607,0.0
7,400608,0.0
8,400610,0.0
9,400612,0.0


In [5]:
baseline = pd.read_csv("data/processed/pincode_baseline_stats.csv")

baseline['iqr_ratio'] = baseline['iqr_monthly_total'] / baseline['median_monthly_total'].replace(0, 1)

baseline.sort_values('iqr_ratio', ascending=False).head(15)


,pincode,median_monthly_total,q1_monthly_total,q3_monthly_total,median_bio,median_demo,median_enroll,iqr_monthly_total,bio_demo_ratio,enroll_demo_ratio,iqr_ratio
51,401603,315.0,156.0,1356.0,315.0,22.0,13.0,1200.0,14.318182,0.590909,3.809524
40,401402,42.0,24.0,110.0,42.0,5.0,1.0,86.0,8.400000,0.200000,2.047619
39,401401,46.0,32.0,126.0,40.0,6.0,0.0,94.0,6.666667,0.000000,2.043478
83,421401,1133.0,949.0,2970.5,1055.0,77.0,33.0,2021.5,13.701299,0.428571,1.784201
67,421101,285.0,196.0,691.0,270.0,27.0,6.0,495.0,10.000000,0.222222,1.736842
53,401605,354.0,151.0,745.0,232.0,51.0,2.0,594.0,4.549020,0.039216,1.677966
87,421502,91.0,49.0,201.5,91.0,0.0,0.0,152.5,91.000000,0.000000,1.675824
52,401604,449.0,126.0,868.0,289.0,22.0,15.0,742.0,13.136364,0.681818,1.652561
49,401601,81.0,62.0,192.0,81.0,7.0,1.0,130.0,11.571429,0.142857,1.604938
36,401303,375.0,313.5,875.0,356.0,0.0,3.0,561.5,356.000000,3.000000,1.497333


Monthly Stream Imbalance 

In [6]:
import pandas as pd

df = pd.read_csv("data/processed/monthly_stream_imbalances.csv")

# ---------- Precision filter ----------
high_confidence = df[
    (df['anom_score'] >= 70) |
    (
        (df['reasons'].str.contains("IQR-based monthly spike", na=False)) &
        (
            df['reasons'].str.contains("ratio", na=False)
        )
    ) |
    (df['bio_demo_ratio_change'] >= 5) |
    (df['enroll_demo_ratio_change'] >= 5)
].copy()

high_confidence.to_csv(
    "data/processed/high_confidence_anomalies.csv",
    index=False
)

print("High-confidence anomalies:", len(high_confidence))


High-confidence anomalies: 464


In [8]:
import pandas as pd
import numpy as np

# ---------- Load flagged rows ----------
flagged_path = "data/processed/monthly_stream_imbalances.csv"
flagged = pd.read_csv(flagged_path)

# ---------- Load baseline ----------
baseline = pd.read_csv("data/processed/pincode_baseline_stats.csv")
# keep only needed baseline columns
baseline_small = baseline[['pincode','median_monthly_total','bio_demo_ratio','enroll_demo_ratio']].copy()

# ---------- Merge baseline into flagged ----------
df = flagged.merge(baseline_small, on='pincode', how='left')

# ---------- Load spike file if exists and merge spike_flag ----------
try:
    spike = pd.read_csv("data/processed/monthly_spike_anomalies.csv")
    spike = spike[['pincode','month']].copy()
    spike['spike_flag'] = True
    df = df.merge(spike, on=['pincode','month'], how='left')
    df['spike_flag'] = df['spike_flag'].fillna(False)
except FileNotFoundError:
    df['spike_flag'] = False

# ---------- Ensure numeric columns exist (create if missing) ----------
eps = 1e-9
if 'bio_updates' not in df.columns or 'demo_updates' not in df.columns:
    raise ValueError("Expected bio_updates and demo_updates in monthly_stream_imbalances.csv")

# current ratios (safe)
df['current_bio_demo_ratio'] = df['bio_updates'] / (df['demo_updates'] + eps)
df['current_enroll_demo_ratio'] = df['enrollments'] / (df['demo_updates'] + eps)

# baseline ratios (safe): fill missing baseline ratio with tiny eps to avoid div-by-zero
df['bio_demo_ratio'] = df['bio_demo_ratio'].fillna(eps)
df['enroll_demo_ratio'] = df['enroll_demo_ratio'].fillna(eps)

# ratio change factors
df['bio_demo_ratio_change'] = df['current_bio_demo_ratio'] / (df['bio_demo_ratio'] + eps)
df['enroll_demo_ratio_change'] = df['current_enroll_demo_ratio'] / (df['enroll_demo_ratio'] + eps)

# compute simple flags if not present
if 'anom_score' not in df.columns:
    df['anom_score'] = 0
if 'reasons' not in df.columns:
    df['reasons'] = ""

# ---------- Define strong conditions (same as discussed) ----------
cond_score = df['anom_score'] >= 70

# spike plus ratio: spike_flag AND (either ratio abnormal by factor >=3)
cond_ratio_above3 = (df['bio_demo_ratio_change'] >= 3) | (df['enroll_demo_ratio_change'] >= 3)
cond_spike_plus_ratio = df['spike_flag'] & cond_ratio_above3

cond_big_ratio = (df['bio_demo_ratio_change'] >= 6) | (df['enroll_demo_ratio_change'] >= 6)

# Need median_monthly_total for big volume test — ensure present
if 'median_monthly_total' not in df.columns:
    # attempt to merge again from baseline if missing
    if 'median_monthly_total' in baseline_small.columns:
        df = df.merge(baseline_small[['pincode','median_monthly_total']], on='pincode', how='left')
    else:
        df['median_monthly_total'] = np.nan

cond_big_volume = df['total_monthly_count'] >= 2 * df['median_monthly_total'].fillna(0)

# ---------- Count how many strong conditions hold ----------
df['strong_count'] = (
    cond_score.astype(int) +
    cond_spike_plus_ratio.astype(int) +
    cond_big_ratio.astype(int) +
    cond_big_volume.astype(int)
)

# ---------- Final filter: require at least 2 strong conditions ----------
final_anomalies = df[df['strong_count'] >= 2].copy()

# ---------- Save final anomalies ----------
out_path = "data/processed/final_anomalies.csv"
final_anomalies.to_csv(out_path, index=False)

# ---------- Print summary ----------
print("FINAL anomalies count:", len(final_anomalies))
print("Saved to:", out_path)

# ---------- Preview first rows for your copy/paste ----------
display_cols = [
    'pincode','month','total_monthly_count','median_monthly_total',
    'anom_score','spike_flag','bio_demo_ratio_change','enroll_demo_ratio_change',
    'strong_count','reasons'
]
display_cols = [c for c in display_cols if c in final_anomalies.columns]
print(final_anomalies[display_cols].head(8).to_string(index=False))


FINAL anomalies count: 24
Saved to: data/processed/final_anomalies.csv
 pincode   month  total_monthly_count  median_monthly_total  anom_score  spike_flag  bio_demo_ratio_change  enroll_demo_ratio_change  strong_count                                                                              reasons
  400610 2025-05                 1086                 508.0          60       False           1.052412e+11              0.000000e+00             2               bio/demo ratio changed 105241237113.4x; enroll/demo ratio changed 0.0x
  401101 2025-11                 3248                1489.0          80        True           1.496780e-01              5.878071e-01             2                                 IQR-based monthly spike; bio/demo ratio changed 0.1x
  401104 2025-11                  149                  37.0         100        True           1.379310e-01              8.163265e-02             2 IQR-based monthly spike; bio/demo ratio changed 0.1x; enroll/demo ratio changed 0.1x
 

C:\Users\Naman\AppData\Local\Temp\ipykernel_21664\2325301001.py:22: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['spike_flag'] = df['spike_flag'].fillna(False)
